# 00 — Preparação dos Dados

Este notebook realiza a preparação inicial dos dados utilizados no Tech Challenge — Fase 2.

## Objetivos

- Obter os dados originais a partir da fonte pública.
- Verificar a estrutura das bases disponíveis.
- Analisar a relação entre os identificadores das bases.
- Estudar o histórico de crédito necessário para definição da variável-alvo.
- Construir e justificar a variável-alvo.
- Consolidar as informações em uma base analítica para as etapas seguintes.

> **Fonte dos dados:** Credit Card Approval Prediction, disponibilizado no Kaggle.

In [34]:
# ============================================================
# 1. CONFIGURAÇÃO DO AMBIENTE
# ============================================================

from pathlib import Path

# Endereço do repositório do projeto
REPO_URL = "https://github.com/andrerochadeoliveira/postech-tech-challenge-fase2-template.git"

# Local onde o repositório ficará disponível no Colab
PROJECT_PATH = Path("/content/postech-tech-challenge-fase2-template")

# Clona o repositório somente se ele ainda não estiver disponível
if not PROJECT_PATH.exists():
    !git clone {REPO_URL}
else:
    print("Repositório já disponível nesta sessão.")

print("\nDiretório do projeto:")
print(PROJECT_PATH)

Repositório já disponível nesta sessão.

Diretório do projeto:
/content/postech-tech-challenge-fase2-template


In [35]:
# ============================================================
# 2. OBTENÇÃO DOS DADOS ORIGINAIS
# ============================================================

import shutil
import kagglehub

# Baixa o dataset ou utiliza a versão já disponível no cache
dataset_path = kagglehub.dataset_download(
    "rikdifos/credit-card-approval-prediction"
)

KAGGLE_PATH = Path(dataset_path)

# Pasta destinada aos dados originais dentro do projeto
RAW_PATH = PROJECT_PATH / "data" / "raw"

# Garante que a pasta exista
RAW_PATH.mkdir(parents=True, exist_ok=True)

# Arquivos originais necessários para o projeto
arquivos = [
    "application_record.csv",
    "credit_record.csv"
]

# Copia os arquivos para data/raw
for nome_arquivo in arquivos:
    origem = KAGGLE_PATH / nome_arquivo
    destino = RAW_PATH / nome_arquivo

    if not destino.exists():
        shutil.copy2(origem, destino)
        print(f"Copiado: {nome_arquivo}")
    else:
        print(f"Já disponível: {nome_arquivo}")

print("\nDados originais disponíveis em:")
print(RAW_PATH)

Using Colab cache for faster access to the 'credit-card-approval-prediction' dataset.
Já disponível: application_record.csv
Já disponível: credit_record.csv

Dados originais disponíveis em:
/content/postech-tech-challenge-fase2-template/data/raw


In [36]:
# ============================================================
# 3. VERIFICAÇÃO DOS ARQUIVOS ORIGINAIS
# ============================================================

for nome_arquivo in arquivos:
    arquivo = RAW_PATH / nome_arquivo

    tamanho_mb = arquivo.stat().st_size / (1024 ** 2)

    print(f"{nome_arquivo}: {tamanho_mb:.2f} MB")

application_record.csv: 51.83 MB
credit_record.csv: 14.66 MB


## 2 — Conhecimento inicial das bases

Nesta etapa, as duas bases originais são carregadas separadamente para análise de sua estrutura, dimensões, variáveis e tipos de dados.

Nenhuma transformação é realizada neste momento.

In [37]:
# ============================================================
# 4. CARREGAMENTO DAS BASES
# ============================================================

import pandas as pd

# Carrega os arquivos originais
df_application = pd.read_csv(
    RAW_PATH / "application_record.csv"
)

df_credit = pd.read_csv(
    RAW_PATH / "credit_record.csv"
)

print("Bases carregadas com sucesso.")

Bases carregadas com sucesso.


In [38]:
# Dimensões das bases
print("application_record:", df_application.shape)
print("credit_record:", df_credit.shape)

application_record: (438557, 18)
credit_record: (1048575, 3)


In [39]:
# ============================================================
# 5. COLUNAS DISPONÍVEIS EM CADA BASE
# ============================================================

print("Colunas de application_record:")
print(df_application.columns.tolist())

print("\nColunas de credit_record:")
print(df_credit.columns.tolist())

Colunas de application_record:
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']

Colunas de credit_record:
['ID', 'MONTHS_BALANCE', 'STATUS']


In [40]:
# ============================================================
# 6. ANÁLISE DOS IDENTIFICADORES
# ============================================================

print("application_record")
print("Linhas:", len(df_application))
print("IDs únicos:", df_application["ID"].nunique())

print("\ncredit_record")
print("Linhas:", len(df_credit))
print("IDs únicos:", df_credit["ID"].nunique())

application_record
Linhas: 438557
IDs únicos: 438510

credit_record
Linhas: 1048575
IDs únicos: 45985


In [41]:
# ============================================================
# 7. RELAÇÃO ENTRE OS IDs DAS DUAS BASES
# ============================================================

# Conjuntos de IDs únicos de cada base
ids_application = set(df_application["ID"])
ids_credit = set(df_credit["ID"])

# IDs presentes nas duas bases
ids_comuns = ids_application.intersection(ids_credit)

print("IDs únicos em application_record:", len(ids_application))
print("IDs únicos em credit_record:", len(ids_credit))
print("IDs presentes nas duas bases:", len(ids_comuns))

IDs únicos em application_record: 438510
IDs únicos em credit_record: 45985
IDs presentes nas duas bases: 36457


In [42]:
# ============================================================
# 8. INVESTIGAÇÃO DE IDs DUPLICADOS EM application_record
# ============================================================

# Identifica todas as linhas cujo ID aparece mais de uma vez
duplicados_application = df_application[
    df_application["ID"].duplicated(keep=False)
].sort_values("ID")

print("Quantidade de linhas com IDs duplicados:",
      len(duplicados_application))

print("Quantidade de IDs envolvidos:",
      duplicados_application["ID"].nunique())

Quantidade de linhas com IDs duplicados: 94
Quantidade de IDs envolvidos: 47


In [43]:
# ============================================================
# 9. VERIFICAÇÃO DOS REGISTROS COM IDs DUPLICADOS
# ============================================================

# Exibe os registros dos primeiros IDs duplicados
duplicados_application.head(10)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
425023,7022197,F,N,Y,0,450000.0,Commercial associate,Higher education,Separated,House / apartment,-19813,-1799,1,0,0,1,NaN,1.0
426818,7022197,M,Y,Y,3,135000.0,Working,Secondary / secondary special,Married,House / apartment,-11945,-735,1,0,0,1,Laborers,5.0
431911,7022327,M,Y,Y,0,256500.0,Commercial associate,Higher education,Married,House / apartment,-21503,-1674,1,0,0,1,Core staff,2.0
431545,7022327,F,N,Y,0,135000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-14771,-5298,1,0,0,0,High skill tech staff,1.0
425486,7023108,M,Y,Y,1,67500.0,Working,Secondary / secondary special,Married,House / apartment,-15156,-1696,1,1,0,0,Core staff,3.0
426488,7023108,F,N,N,0,135000.0,Working,Secondary / secondary special,Married,House / apartment,-17590,-1273,1,0,0,0,Cleaning staff,2.0
425306,7023651,F,N,N,0,225000.0,Commercial associate,Incomplete higher,Single / not married,House / apartment,-10229,-1209,1,0,0,0,Accountants,1.0
421907,7023651,M,Y,N,1,157500.0,Commercial associate,Incomplete higher,Married,House / apartment,-10521,-1457,1,0,0,0,Drivers,3.0
427778,7024111,M,N,N,2,157500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-15270,-117,1,1,0,0,Drivers,4.0
432643,7024111,F,N,Y,0,180000.0,Working,Secondary / secondary special,Married,House / apartment,-22041,-1524,1,1,1,0,Accountants,2.0


In [44]:
# ============================================================
# 10. IDs DUPLICADOS QUE POSSUEM HISTÓRICO DE CRÉDITO
# ============================================================

# IDs que aparecem mais de uma vez na base cadastral
ids_duplicados = set(duplicados_application["ID"])

# Verifica quais deles também aparecem na base de crédito
ids_duplicados_com_credito = ids_duplicados.intersection(ids_credit)

print("IDs duplicados em application_record:",
      len(ids_duplicados))

print("IDs duplicados que também aparecem em credit_record:",
      len(ids_duplicados_com_credito))

IDs duplicados em application_record: 47
IDs duplicados que também aparecem em credit_record: 0


### Análise dos identificadores

A base `application_record` possui 438.557 registros e 438.510 IDs únicos. Foram identificados 47 IDs repetidos, cada um associado a dois registros cadastrais distintos.

A inspeção desses casos mostrou que não se tratam de duplicatas exatas, pois existem diferenças entre as características cadastrais associadas ao mesmo identificador.

Entretanto, nenhum dos 47 IDs duplicados está presente na base `credit_record`. Dessa forma, esses registros não pertencem ao conjunto de clientes para os quais existe histórico de crédito e, consequentemente, não integrarão a base analítica utilizada para construção da variável-alvo e posterior modelagem.

As bases possuem 36.457 IDs em comum.

## 3 — Análise do histórico de crédito

A base `credit_record` contém o histórico mensal de pagamento dos clientes.

Nesta etapa, serão analisadas as variáveis `MONTHS_BALANCE` e `STATUS`, com foco no período utilizado para a construção da variável-alvo.

Para garantir comparabilidade entre os clientes, a população de modelagem será composta somente por clientes que possuem **seis meses consecutivos de histórico**, correspondentes aos meses `0, -1, -2, -3, -4 e -5`.

A variável-alvo será construída a partir do comportamento observado nessa janela de seis meses.

In [45]:
# ============================================================
# 11. PRIMEIROS REGISTROS DO HISTÓRICO DE CRÉDITO
# ============================================================

df_credit.head(10)

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C
5,5001712,-1,C
6,5001712,-2,C
7,5001712,-3,C
8,5001712,-4,C
9,5001712,-5,C


In [46]:
# ============================================================
# 12. DISTRIBUIÇÃO DA VARIÁVEL STATUS
# ============================================================

# Contagem de registros para cada situação de crédito
status_counts = (
    df_credit["STATUS"]
    .value_counts()
    .sort_index()
)

print(status_counts)

STATUS
0    383120
1     11090
2       868
3       320
4       223
5      1693
C    442031
X    209230
Name: count, dtype: int64


In [47]:
# Percentual de cada categoria de STATUS
status_percentual = (
    df_credit["STATUS"]
    .value_counts(normalize=True)
    .mul(100)
    .sort_index()
    .round(2)
)

print(status_percentual)

STATUS
0    36.54
1     1.06
2     0.08
3     0.03
4     0.02
5     0.16
C    42.16
X    19.95
Name: proportion, dtype: float64


In [48]:
# ============================================================
# 13. QUANTIDADE DE MESES OBSERVADOS POR CLIENTE
# ============================================================

meses_por_cliente = (
    df_credit
    .groupby("ID")
    .size()
)

print(meses_por_cliente.describe())

count    45985.000000
mean        22.802544
std         15.492771
min          1.000000
25%         10.000000
50%         19.000000
75%         34.000000
max         61.000000
dtype: float64


In [49]:
# ============================================================
# 14. INTERVALO TEMPORAL DO HISTÓRICO
# ============================================================

print("Maior valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].max())

print("Menor valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].min())

print("Quantidade de valores distintos:",
      df_credit["MONTHS_BALANCE"].nunique())

Maior valor de MONTHS_BALANCE: 0
Menor valor de MONTHS_BALANCE: -60
Quantidade de valores distintos: 61


# 4 — Clientes com histórico
Filtrar apenas clientes com, pelo menos, 6 meses de histórico. Essa definição segue práticas já disseminadas no mercado. Serão considerados apenas os cliente com todos os 6 primeiros meses de histórico

**Opção 2**: Filtrar os cliente com 6 meses de histórico e aplicar a seguinte regra para clientes maus pagadores (target = 1):  
- ao menos 1 (uma) ocorrência de atraso >= 60 dias nos últimos 6 meses  
- ao menos 2 ocorrências de a atraso >= 30 dias nos últimos 6 meses

In [59]:
# Filtrar registros entre os meses -6 e 0
df_meses_alvo = df_credit[df_credit['MONTHS_BALANCE'].between(-6, 0)]

# Contar quantos meses únicos cada cliente tem nesse intervalo
contagem_meses = df_meses_alvo.groupby('ID')['MONTHS_BALANCE'].nunique()

# Identificar os IDs que possuem exatamente os 7 meses
ids_historico_completo = contagem_meses[contagem_meses == 7].index

# 4. Filtrar o df_credit original mantendo apenas esses clientes com histórico completo
df_credit_ftrd = df_credit[df_credit['ID'].isin(ids_historico_completo)].copy()

# Validação do filtro
print(f"Clientes originais no histórico: {df_credit['ID'].nunique()}")
print(f"Clientes com histórico COMPLETO de -6 a 0: {df_credit_ftrd['ID'].nunique()}")
print(f"Total de registros mantidos: {len(df_credit_ftrd)}")

Clientes originais no histórico: 45985
Clientes com histórico COMPLETO de -6 a 0: 29420
Total de registros mantidos: 811671


In [64]:
# DEFINIÇÃO DO TARGET

# Filtrar registros entre os meses -12 e 0 para a análise do target
df_target_period = df_credit_ftrd[df_credit_ftrd['MONTHS_BALANCE'].between(-12, 0)]

# 1. Ao menos 1 atraso >= 90 dias (STATUS: '3', '4', '5')
cond_90d = df_target_period['STATUS'].isin(['3', '4', '5'])
ids_atraso_90d = df_target_period[cond_90d]['ID'].unique()

# 2. Ao menos 2 atrasos >= 30 dias (STATUS: '1', '2', '3', '4', '5')
cond_30d = df_target_period['STATUS'].isin(['1', '2', '3', '4', '5'])
contagem_30d = df_target_period[cond_30d].groupby('ID').size()
ids_atraso_30d = contagem_30d[contagem_30d >= 2].index

# União dos IDs elegíveis para o target como 1 (Mau Pagador)
ids_mau_pagador = set(ids_atraso_90d).union(set(ids_atraso_30d))

# Criação do DataFrame de target para todos os IDs únicos em df_credit_ftrd
df_target = pd.DataFrame({'ID': df_credit_ftrd['ID'].unique()})
df_target['target'] = df_target['ID'].apply(lambda x: 1 if x in ids_mau_pagador else 0)

# Exibindo a distribuição e proporção do novo target
print("Distribuição do novo target:")
print(df_target['target'].value_counts())
print("\nPercentual:")
print(df_target['target'].value_counts(normalize=True).mul(100).round(2))

Distribuição do novo target:
target
0    28492
1      928
Name: count, dtype: int64

Percentual:
target
0    96.85
1     3.15
Name: proportion, dtype: float64


# 6 - Join com a tabela da cadastro
Fazer merge com a tabela de cadastro adicionando o atributo "target" para cada cliente

In [65]:
# Realiza a junção interna (inner join) entre df_application e df_target utilizando a coluna 'ID'
df_application_target = df_application.merge(df_target, on='ID', how='inner')

# Exibe informações sobre as dimensões gerais
print(f"Dimensões de df_application_target: {df_application_target.shape}")
print(f"Quantidade de IDs únicos: {df_application_target['ID'].nunique()}")

# Calcula e exibe a quantidade e proporção de clientes por classe do target (bad)
contagem_bad = df_application_target['target'].value_counts()
percentual_bad = df_application_target['target'].value_counts(normalize=True).mul(100).round(2)

print("\nQuantidade de clientes por classe (target):")
for classe, qtd in contagem_bad.items():
    pct = percentual_bad[classe]
    print(f"Classe {classe}: {qtd} clientes ({pct}%)")

print("\nPrimeiros registros do DataFrame unificado:")
display(df_application_target.head())

Dimensões de df_application_target: (20929, 19)
Quantidade de IDs únicos: 20929

Quantidade de clientes por classe (target):
Classe 0: 20192 clientes (96.48%)
Classe 1: 737 clientes (3.52%)

Primeiros registros do DataFrame unificado:


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,target
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0
3,5008810,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0
4,5008811,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0


## 7 — Exportação da base analítica
A base final de modelagem, composta pelas informações cadastrais e pela variável-alvo construída a partir do histórico de crédito, é exportada para utilização nas etapas seguintes do projeto.

In [66]:
# ============================================================
# 34. EXPORTAÇÃO DA BASE ANALÍTICA
# ============================================================

# Pasta destinada às bases processadas
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"

# Garante que a pasta exista
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Caminho da base que será utilizada pelo Notebook 01
DATASET_BASE_PATH = PROCESSED_PATH / "dataset_base.csv"

# Exporta a base final de modelagem
df_application_target.to_csv(
    DATASET_BASE_PATH,
    index=False
)

print(f"Arquivo: {DATASET_BASE_PATH}")

Arquivo: /content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv
